# 上一交易日 SOXX 回撤驱动的 Stage1 风险钟：只读必要机制预检

机械复现原0bp账户风险状态；使用决策前已知的拆分连续 SOXX Qlib 前日收盘，不构造资金反事实。初步预览已披露在 plan.json。


In [ ]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.workflow.intraday_recovery_direct_allocation import _advance_state,_base_cap,IntradayRecoveryDirectAllocationSpec
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_soxx_lagged_drawdown_risk_preflight_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
parent=root/src['parent_card'];comp=root/src['component_card'];prec=parent/'qlib_recorder'/src['parent_experiment_id']/src['parent_recorder_id']/'artifacts';crec=comp/'qlib_recorder'/src['component_experiment_id']/src['component_recorder_id']/'artifacts'
for p,h in [(prec/'paths/A_active_research_replay_0',src['parent_path_sha256']),(prec/'inputs/frame',src['parent_frame_sha256']),(prec/'inputs/raw_points',src['parent_raw_points_sha256']),(prec/'inputs/recovery_spec',src['parent_recovery_spec_sha256']),(crec/'analysis/daily',src['component_daily_sha256']),(root/src['provider_uri']/'qlib_store_manifest.json',src['provider_manifest_sha256']),(root/'src/workflow/intraday_recovery_direct_allocation.py',src['risk_module_sha256'])]:assert sha(p)==h,(p,sha(p),h)
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['provider_uri']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2023-01-03',end_time='2026-10-07',freq='day'));raw=D.features(['SOXX'],['$close'],start_time='2023-01-03',end_time='2026-10-07',freq='day');price=raw['$close'].droplevel('instrument').astype(float);assert price.index.equals(cal) and np.isfinite(price).all() and price.gt(0).all()
with R.uri_context(str(parent/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['parent_experiment_id'],recorder_id=src['parent_recorder_id']);assert rec.status=='FINISHED';path=rec.load_object('paths/A_active_research_replay_0');frame=rec.load_object('inputs/frame');raw_points=rec.load_object('inputs/raw_points');spec=IntradayRecoveryDirectAllocationSpec(root=root,**rec.load_object('inputs/recovery_spec'))
with R.uri_context(str(comp/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['component_experiment_id'],recorder_id=src['component_recorder_id']);assert rec.status=='FINISHED';components=rec.load_object('analysis/daily')
assert len(frame)==len(path['replay'])==len(path['targets'])==len(path['trace'])==len(components)==944
assert pd.DatetimeIndex(frame.signal_date).equals(price.index) and pd.DatetimeIndex(frame.trade_date).equals(pd.DatetimeIndex(components.trade_date))
split_preview={'before_split_raw_0945':float(raw_points.loc[pd.Timestamp('2024-03-06 09:45'),'SOXX']),'after_split_raw_0945':float(raw_points.loc[pd.Timestamp('2024-03-07 09:45'),'SOXX']),'before_split_qlib_prior_close':float(price.loc[pd.Timestamp('2024-03-06')]),'after_split_qlib_prior_close':float(price.loc[pd.Timestamp('2024-03-07')])};assert split_preview['before_split_raw_0945']/split_preview['after_split_raw_0945']>2.5 and abs(split_preview['before_split_qlib_prior_close']/split_preview['after_split_qlib_prior_close']-1)<0.1
benchmark_drawdown=(1-price/price.cummax()).clip(lower=0);state=clock=exo_state=exo_clock=0;history=[];previous=None;daily=[]
for i,row in frame.iterrows():
 trade=pd.Timestamp(row.trade_date);signal=pd.Timestamp(row.signal_date);own=path['targets'].iloc[i];trace=path['trace'].iloc[i];known_nav=float(trace.known_nav)
 if previous is not None:history.append(known_nav/previous-1)
 previous=known_nav;own_dd=float(own.known_drawdown);state,clock,q=_advance_state(state,clock,own_dd,spec);own_cap,_=_base_cap(row,q=q,drawdown=own_dd,history=history,spec=spec)
 assert state==int(trace.risk_state) and clock==int(trace.risk_clock) and abs(own_cap-float(trace.current_beta_cap))<1e-11
 exo_dd=float(benchmark_drawdown.loc[signal]);exo_state,exo_clock,exo_q=_advance_state(exo_state,exo_clock,exo_dd,spec);exo_cap,_=_base_cap(row,q=exo_q,drawdown=exo_dd,history=history,spec=spec)
 c=components.iloc[i];daily.append({'signal_date':signal,'trade_date':trade,'own_drawdown':own_dd,'SOXX_lagged_close_drawdown':exo_dd,'own_state':state,'SOXX_clock_state_fixed_path':exo_state,'own_cap':own_cap,'SOXX_clock_cap_fixed_path':exo_cap,'exogenous_minus_own_cap':exo_cap-own_cap,'cash_opportunity_log':float(c.cash_opportunity_log),'SOXL_selection_log':float(c.SOXL_selection_log),'paired_log_excess':float(c.paired_log_excess)})
daily=pd.DataFrame(daily);assert len(daily)==944 and daily.trade_date.is_unique and daily.signal_date.is_monotonic_increasing
static=pd.concat({'feature':daily[['SOXX_lagged_close_drawdown']],'meta':daily[['own_drawdown','own_cap','SOXX_clock_cap_fixed_path','cash_opportunity_log']]},axis=1);static.index=pd.MultiIndex.from_arrays([daily.trade_date,['SOXL_SOXX_CASH']*len(daily)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'evaluation':('2023-01-04','2026-10-08')});assert len(dataset.prepare('evaluation',col_set=['feature','meta']))==944
def describe(g):
 neg=g.loc[g.cash_opportunity_log.lt(0)];return {'sessions':len(g),'state_differences':int(g.own_state.ne(g.SOXX_clock_state_fixed_path).sum()),'cap_difference_gt_0p05':int(g.exogenous_minus_own_cap.abs().gt(.05).sum()),'mean_exogenous_minus_own_cap':float(g.exogenous_minus_own_cap.mean()),'cash_opportunity_log_sum':float(g.cash_opportunity_log.sum()),'SOXL_selection_log_sum':float(g.SOXL_selection_log.sum()),'negative_cash_days':len(neg),'negative_cash_mean_cap_difference':float(neg.exogenous_minus_own_cap.mean()) if len(neg) else None,'negative_cash_days_with_higher_cap':int(neg.exogenous_minus_own_cap.gt(1e-8).sum())}
annual={str(y):describe(g) for y,g in daily.groupby(daily.trade_date.dt.year)};quarter={str(q):describe(g) for q,g in daily.groupby(daily.trade_date.dt.to_period('Q'))};rolling=[]
for i in range(len(daily)-125):
 g=daily.iloc[i:i+126];rolling.append({'first_trade':str(g.trade_date.iloc[0].date()),'last_trade':str(g.trade_date.iloc[-1].date()),**describe(g)})
assert len(annual)==4 and len(quarter)==16 and len(rolling)==819
result={'status':'read_only_lagged_SOXX_drawdown_clock_fixed_original_path_mechanism_mismatch','plan_sha256':sha(card/'plan.json'),'split_basis_preview':split_preview,'all':describe(daily),'annual':annual,'quarterly':quarter,'rolling126_count':len(rolling),'original_state_and_cap_parity':'944/944 exact within1e-11','Qlib_dataset_rows':944,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
statefile={'status':'Recorder_started','plan_sha256':result['plan_sha256'],'new_model_fits':0,'new_native_accounts':0};(card/'run_state.json').write_text(json.dumps(statefile,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset,split_basis_preview=split_preview);rec.save_objects(artifact_path='analysis',daily=daily,annual=annual,quarterly=quarter,rolling126=pd.DataFrame(rolling),result=result);rec.log_metrics(sessions=944,own_parity_days=944,exogenous_state_difference_days=result['all']['state_differences'],new_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
statefile['status']='finished';statefile['recorder_id']=rid;statefile['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(statefile,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);remote=_ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
summary={'experiment':plan['experiment'],'decision':'closed_before_account_fixed_lagged_SOXX_drawdown_clock_cash_capacity_mismatch','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':remote,'verified':True},'new_model_fits':0,'new_native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/annual','analysis/quarterly','analysis/rolling126','analysis/result']},'limitations':['The exogenous state/cap is recomputed on the frozen original own-history path; no funded candidate feedback is simulated.','The earlier scratch preview is disclosed and these development intervals are not independent.','Lagged Qlib split-continuous SOXX close is observable before 09:46; current-day raw-price peak is invalid across the 2024 split.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':summary['decision'],'all':result['all'],'2024Q3':quarter['2024Q3'],'2025':annual['2025'],'2026Q3':quarter['2026Q3'],'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False,indent=2))


## 中文结论

对冻结完整 Stage1 的 **944 日0bp、仍计 Tiered 费用**原账户，按同一风险规格逐日重新执行 `_advance_state` 与 `_base_cap`；自身状态/上限与原生Recorder在 **944/944** 日一致。候选坐标只用决策前已知的上一交易日、Qlib拆分连续 SOXX 收盘价相对其历史峰值的回撤；原账户自身的已知收益历史暂固定，故只读比较状态输入的传递，**不是新资金账户**。全部4年度、16季度与819个连续126日窗口保存在新原生Recorder。

原始09:45点价不能直接做跨2024拆分的高水位：SOXX在2024-03-06/07的原价约 **689.45/234.12**，而Qlib拆分连续前日收盘约 **226.57/234.35**。已在审查前排除原价误差；不把其虚假的约三分之二回撤当市场风险信号。

更换风险钟在原路径上造成 **356** 日状态不同、**253** 日上限差超过0.05；平均外生减原上限 **-0.0079**。核心差区间并未得到所需容量：2025原账户现金机会对数贡献 **-0.15801**；141个现金分量为负的日子中，仅 **7** 日外生钟提高上限，平均上限差 **-0.0917**。2024Q3现金保护为 **+0.06444**，外生钟平均压低上限 **-0.0979**；2026Q3外生钟反而提高上限 **+0.1417**，同期原账户 SOXL相对选择对数损失 **-0.14390**。这些区间用于解释必要机制，不是按年份交易的规则。

决定 **`closed_before_account_fixed_lagged_SOXX_drawdown_clock_cash_capacity_mismatch`**：这一条固定的前日SOXX回撤替代路径在2025广泛现金拖累层方向相反，停止在机制预检，不追加0bp资金账户，也不改滞后、价格口径、拆分、阈值或周期救援。外生风险状态会改写真实账户后续持仓、费用、净值与自身收益历史；本只读比较**不能证明任何完整反事实必然亏损**。前期预览已在计划披露，复用2023–2026历史不构成独立验证。

Qlib Recorder `3cee6634c9d548408df987f67882348d`，NAS SHA-256 `39a4e59acc5e5b1f0acc3a4ffe0299aa02f8800dae9f91d2af28ac517af243c1`，五件日级/年度/季度/滚动/结果工件经远端限定冷读逐字节一致。0拟合、0账户、0bootstrap；SOXX稳定超额、2/5bp及独立前向门仍未通过。
